In [19]:
import torch
import pandas as pd

In [21]:
df = pd.read_csv("../data/processed/final_rep_virus_dataset.csv")

In [22]:
df

,taxid,virus_name,sDBM_pos,sDBM,sDBM_sim_gap,ori_seq,ori_align_score,iteration_name,rep_amino_acid_seq,match
0,59817,Faba bean necrotic yellows virus,"(63, 71)",GAHFEKRRG,33.0,TAGTATTAC,138.5,FBNYY_iter1_V_A,MARQVICWCFTLNNPLSPLSLHDSMKYLVYQTEQGEAGNIHFQGYI...,Yes
1,262588,Cardamom bushy dwarf virus,"(63, 71)",GAHLEKRKG,33.1,TATTATTAC,138.2,FBNYY_iter1_V_A,MARYVVCWMFTINNPSSLPVMRDEIKYMVYQVERGQEGTRHVQGYV...,Yes
2,1260769,Colecusatellite agerati,"(73, 81)",KPHLEKQRA,22.3,GACTGCTAC,130.0,FBNYY_iter1_V_A,MAAIKGQWWCFTVFFLTATAPDLVPLFENTHVSYACWQEEESPTTK...,Yes
3,2169722,Banana bunchy top alphasatellite 2,"(70, 78)",RAHWEKARG,40.3,TAGTATTAC,138.5,FBNYY_iter1_V_A,MSASRWTFTLHYSDATERGKFLATLKEEDVHYAVVGDETAPNTGRK...,Yes
4,1685737,Lake Sarah-associated circular virus-11,"(74, 82)",RAHFECAKG,40.2,CAGTATTAC,137.7,FBNYY_iter1_V_A,MAPRGNAAKRWTFTINNWTLADYDKVLEFTTKFEPVFLICGKERGE...,Yes
...,...,...,...,...,...,...,...,...,...,...
471,222450,Chayote yellow mosaic virus,"(88, 96)",HPNVQGAKS,37.7,TGATGTTAC,132.4,SGC1_iter2_V_R,MAPPKRFQINAKNYFLTYPQCSLTKEEALSQLRSLRTPTNQKYIKV...,No
472,223332,Tobacco leaf curl Japan virus,"(88, 96)",HPNIQGAKS,37.9,TATTACTAC,139.3,SGC1_iter2_V_R,MAPPKRFQINAKNYFLTYPHCSLTKEEALSQIQALDTITNKLFIRI...,No
473,486809,Ramie mosaic virus,"(88, 96)",HPNIQGAKS,37.4,TAGTATTAC,139.3,SGC1_iter2_V_R,MAPPRRFTINAKNYFLTYPQCPLSKEEALSQLLNLSTPTNKKFIKI...,No
474,390437,Kudzu mosaic virus,"(88, 96)",HPNIQGAKS,37.0,TAGTATTAC,130.6,SGC1_iter2_V_R,MSRPKGFRVNAKNFFLTYPRCSLSKEAALEQLQNIQTNVNKKFIRV...,No


In [23]:
df_dropped = df[['sDBM_pos', 'ori_seq', 'rep_amino_acid_seq', 'match']]

df_dropped

,sDBM_pos,ori_seq,rep_amino_acid_seq,match
0,"(63, 71)",TAGTATTAC,MARQVICWCFTLNNPLSPLSLHDSMKYLVYQTEQGEAGNIHFQGYI...,Yes
1,"(63, 71)",TATTATTAC,MARYVVCWMFTINNPSSLPVMRDEIKYMVYQVERGQEGTRHVQGYV...,Yes
2,"(73, 81)",GACTGCTAC,MAAIKGQWWCFTVFFLTATAPDLVPLFENTHVSYACWQEEESPTTK...,Yes
3,"(70, 78)",TAGTATTAC,MSASRWTFTLHYSDATERGKFLATLKEEDVHYAVVGDETAPNTGRK...,Yes
4,"(74, 82)",CAGTATTAC,MAPRGNAAKRWTFTINNWTLADYDKVLEFTTKFEPVFLICGKERGE...,Yes
...,...,...,...,...
471,"(88, 96)",TGATGTTAC,MAPPKRFQINAKNYFLTYPQCSLTKEEALSQLRSLRTPTNQKYIKV...,No
472,"(88, 96)",TATTACTAC,MAPPKRFQINAKNYFLTYPHCSLTKEEALSQIQALDTITNKLFIRI...,No
473,"(88, 96)",TAGTATTAC,MAPPRRFTINAKNYFLTYPQCPLSKEEALSQLLNLSTPTNKKFIKI...,No
474,"(88, 96)",TAGTATTAC,MSRPKGFRVNAKNFFLTYPRCSLSKEAALEQLQNIQTNVNKKFIRV...,No


In [5]:
from esm.models.esmc import EsmcForMaskedLM, EsmcTokenizer
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ESMC: Transformer Engine is not installed; falling back to pure-PyTorch LayerNorm+Linear / LayerNorm+MLP. Outputs will differ numerically - measured on the unnormalized residual stream (before the final LayerNorm), ~O(10) max-diff in fp32 and ~O(100) in bf16; after the final LayerNorm these shrink to a few ULP and perplexity stays within rounding noise. Install with `pip install transformer-engine[pytorch]` to enable fused fp32-reduction LayerNorm.
ESMC: neither xformers nor flash-attn is installed; falling back to PyTorch ``F.scaled_dot_product_attention``. The attention reduction order in bf16 differs from a fused kernel by ~1 bf16 ULP per attention block; compounded across the 80-block stack this reaches ~O(100) max-diff on the unnormalized residual stream. Install an xformers build matching your PyTorch and CUDA versions for a fused attention kernel.


In [ ]:
# ESMC-300M Pretrained model
model = EsmcForMaskedLM.from_pretrained(
    "biohub/ESMC-300M",
    device=device,
).eval()

tokenizer = EsmcTokenizer()

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

In [ ]:
def get_protein_embedding(amino_acid_seq):

    inputs = tokenizer([amino_acid_seq], return_tensors="pt", padding=True)
    inputs = {name: value.to(model.device) for name, value in inputs.items()}

    with torch.inference_mode():
        result = model(**inputs, output_hidden_states=True)

    token_embeddings = result.hidden_states[-1]    
    residue_embeddings = token_embeddings[0, 1:-1] # exlude start and end token

    return residue_embeddings

In [ ]:
def get_dna_one_hot_encoding(ori_seq):

    vocab = "ACGT"

    # character to index lookup matching
    base_to_idx = {char: idx for idx, char in enumerate(vocab)}

    # convert ori_seq -> indices
    indices = [base_to_idx.get(base, base_to_idx.get(' ', 0)) for base in ori_seq]

    #convert to tensor
    index_tensor = torch.tensor(indices, dtype=torch.long)

    one_hot = F.one_hot(index_tensor, num_classes=4)

    return one_hot

In [6]:
import ast

from sklearn.preprocessing import StandardScaler

In [ ]:
sDBM_tensor_list = []

ori_tensor_list = []


for index, row in df_dropped.iterrows():

    sDBM_start, sDBM_end = ast.literal_eval(row.values[0]) #tuple exists as a string -> convert str to tuple
    sDBM_end += 1 


    ori_seq = row.values[1]
    amino_seq = row.values[2]

    sDBM_embedding = get_protein_embedding(amino_seq)[sDBM_start : sDBM_end, :]
    ori_one_hot = get_dna_one_hot_encoding(ori_seq)

    sDBM_tensor_list.append(sDBM_embedding)
    ori_tensor_list.append(ori_one_hot)


embedding_tensor = torch.stack(sDBM_tensor_list, dim = 0)
ori_one_hot_tensor = torch.stack(ori_tensor_list, dim = 0)

In [14]:
scaleStandard = StandardScaler()
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

import random

pca1 = PCA()

In [3]:
# temporary
embedding_tensor = torch.rand(5, 9, 960)
ori_one_hot_tensor = torch.rand(5, 9, 4)

In [33]:
ori_one_hot = ori_one_hot_tensor.flatten(-2)
ori_one_hot.shape

torch.Size([5, 36])

In [4]:
embed_np = embedding_tensor.cpu().numpy()
ori_np = ori_one_hot_tensor.cpu().numpy()

In [16]:
pca2 = PCA(0.95)

reduced_residue_embeddings = []

for i in range(0, 9):

    residue_embed = embed_np[:, i, :]

    residue_embed_scaled = scaleStandard.fit_transform(residue_embed)

    residue_embed_pca2 = pca2.fit_transform(residue_embed_scaled)

    rand_num = random.randint(0, 37)
    residue_embed_pca2 = torch.rand(5, rand_num)
    reduced_residue_embeddings.append(residue_embed_pca2)

    #reduced_residue_embeddings.append(torch.from_numpy(residue_embed_pca2))

    print(residue_embed_pca2.shape)

sDBM_embedding = torch.concat(reduced_residue_embeddings, dim = 1)

print(sDBM_embedding.shape)

torch.Size([5, 1])
torch.Size([5, 7])
torch.Size([5, 25])
torch.Size([5, 18])
torch.Size([5, 28])
torch.Size([5, 19])
torch.Size([5, 14])
torch.Size([5, 1])
torch.Size([5, 12])
torch.Size([5, 125])


In [ ]:
binary_target_list = []


for index, row in df_dropped.iterrows():

    match_str = row.values[3]

    if match_str == 'Yes':
        binary_target_list.append(1)
    else:
        binary_target_list.append(0)


targets_tensor = torch.tensor(binary_target_list, dtype = torch.float32)

#targets_tensor.unsqueeze(1).shape


torch.Size([476, 1])

In [31]:
print(targets_tensor.shape)
print(targets_tensor)

torch.Size([476])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0,

In [ ]:
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split

In [ ]:
class CustomDataset(Dataset):
    # required dunder methods
    def __init__(self, data, target):
        self.data = torch.tensor(data, dtype = torch.float32)
        self.target = torch.tensor(target, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        sample = {'data': self.data[index], 'target': self.target[index]}
        return sample

In [ ]:

# # 2. Wrap them in a TensorDataset
# dataset = TensorDataset(embeddings, targets)

# # 3. Define your split sizes (e.g., 80% train, 20% test)
# train_size = int(0.8 * len(dataset))
# test_size = len(dataset) - train_size

# # 4. Split the dataset
# train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

In [ ]:
sDBM_embeddings = torch.load("sDBM_embeddings.pt")
ori_one_hot = torch.load("ori_one_hot.pt")
targets = torch.load("target.pt")